# This notebook extracts realizations to be used for CMG simulations

* The finit files must contain ACTID to identify active cells, in addition to properties such as PORO and PERMX.
* This notebook has been validated by (1) processing a realization using this notebook, (2) import processed realiztion into Petrel, (3) compare it with the realization in Petrel and found statistics are exactly the same. Note when exporting a property out of Petrel, setting undefined cells to be 0 (instead of -999) would generate the exact same file as a processed realization by this notebook.
* For CMG, set reverse_j = True when using generate_full_properties().
* For Vienna geothermal, step 1 extracts PORO and PERMX from the finit files and the initial temperature TEMP from the f0000 files.
* (2026-10-09) The finit and f0000 files are read directly from each realization folder; they are no longer copied first. Copies made over the Z: (Oak SMB) mount were found with 1 MiB blocks replaced by NUL bytes, which caused 25 realizations to be skipped (see `03_results/2026-10-09_finit_integrity/README.md`). The realizations in `01_data/geothermal_Vienna/raw/pre_Petrel_realizations` were recopied on Sherlock and verified clean on 2026-10-09. `extract_properties_from_finit` now rejects files with NUL bytes and checks every array against the size declared in its header.
* (2026-10-10) Petrel exported the same f0000 TEMP for every realization (11 + 0.0315 * (Z + 155), capped at 137 C), ignoring the sampled `Temp_grad` (see `03_results/2026-10-09_property_verification/README.md`). Optional step 2 replaces the TEMP files with TEMP = Temp_grad * Z + 11 for every cell, using each realization's sampled `Temp_grad`.

## Step 1: Extract properties

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root / '02_code' / 'src'))

from pre_extract_properties_from_finit import extract_properties_from_finit
from pre_generate_full_properties import generate_full_properties
from pre_CMG_format_compress import CMG_format_compress

data_root = project_root / '01_data' / 'geothermal_Vienna'

################# Start of user inputs ##########
# folder containing one subfolder per realization, each with a .FINIT and a .F0000 file
# note: copy realizations on Sherlock (rsync -avc), not over the Z: mount, which corrupts large files (see 03_results/2026-10-09_finit_integrity)
realizations_folder_path = data_root / 'raw' / 'pre_Petrel_realizations'
save_folder_path = data_root / 'processed' / 'pre_properties' # path to the folder where extracted properties will be saved. Will be created if it doesn't exist.
log_file_path = data_root / 'processed' / 'pre_property_file_names' / '261009_realization_property_extract_log.txt' # messages are also written here
grid_shape = (139, 248, 23) # shape of the geomodel grid (i, j, k)
finit_property_list = ['PORO', 'PERMX']  # list of property keywords to extract from .FINIT files
f0000_property_list = ['TEMP']  # list of property keywords to extract from .F0000 files
case_list = None # e.g. [125, 126] to process only these realizations; None processes all
skip_existing = True # skip realizations whose property files already exist in save_folder_path
################## End of user inputs  ##########

save_folder_path.mkdir(exist_ok=True)
property_list = finit_property_list + f0000_property_list

def extract_number(name):
    return int(name.split('_')[-1])

case_folders = sorted([p for p in realizations_folder_path.iterdir() if p.is_dir()], key=lambda p: extract_number(p.name))
if case_list is not None:
    case_folders = [p for p in case_folders if extract_number(p.name) in case_list]

log = open(log_file_path, 'a')
def report(message):
    print(message)
    log.write(message + '\n')

report(f'Extracting {property_list} from {len(case_folders)} realizations in {realizations_folder_path}')
n_done, n_skipped_existing, failed = 0, 0, []
for case_folder in case_folders:
    case_name = case_folder.name
    finit_file_path = case_folder / f'{case_name}.FINIT'
    f0000_file_path = case_folder / f'{case_name}.F0000'

    if skip_existing and all((save_folder_path / f'{case_name}_{key}.dat').exists() for key in property_list):
        n_skipped_existing += 1
        continue

    # corrupted or incomplete files raise ValueError/KeyError, so skip them and record the reason
    try:
        for file_path in (finit_file_path, f0000_file_path):
            if not file_path.exists():
                raise ValueError(f'{file_path.name} not found (failed Petrel realization?)')

        # STEP 1: Extract properties from FINIT file for active cells only
        extracted_property_dict = extract_properties_from_finit(
            finit_file_path = finit_file_path,
            keywords = finit_property_list + ['ACTID'],
            is_save = False,
            save_dir = save_folder_path,
            save_name = case_name,
            show_summary = False
        )
        # Extract properties from F0000 file for active cells only
        extracted_property_dict_F0000 = extract_properties_from_finit(
            finit_file_path = f0000_file_path,
            keywords = f0000_property_list,
            is_save = False,
            save_dir = save_folder_path,
            save_name = case_name,
            show_summary = False
        )
        # Add the temperature property from the F0000 file to the property dictionary
        extracted_property_dict.update(extracted_property_dict_F0000)

        # STEP 2: Generate full properties for all cells (fill inactive cells with zeros)
        full_property_dict = generate_full_properties(
            property_dict = extracted_property_dict,
            property_list = property_list,
            grid_shape = grid_shape,
            is_save = False,
            save_dir = save_folder_path,
            save_name = case_name,
            show_summary = False,
            reverse_j = True
            )

        # STEP 3: Compress full properties to CMG format (repeated values as N*value)
        for key in property_list:
            CMG_format_compress(
                array = full_property_dict[key],
                keyword = key,
                max_line_length = 80,
                show_summary = False,
                save_dir = save_folder_path,
                save_name = case_name
            )
            log.write(f'Saved compressed {key} data to: {save_folder_path / f"{case_name}_{key}.dat"}\n')
        n_done += 1
    except (ValueError, KeyError) as e:
        failed.append(case_name)
        report(f"Warning: Skipping '{case_name}'. Reason: {e}")
        continue

report(f'Done: {n_done} extracted, {n_skipped_existing} skipped (already extracted), {len(failed)} failed: {failed}')
log.close()

## Step 2 (optional): Generate TEMP from the sampled temperature gradient
Replaces the TEMP files from step 1 with TEMP = Temp_grad * Z + 11, calculated for every cell.

In [1]:
import sys
import numpy as np
from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root / '02_code' / 'src'))

from pre_generate_temperature import load_temp_gradients, load_cell_depths, temperature_from_gradient
from pre_CMG_format_compress import CMG_format_compress

data_root = project_root / '01_data' / 'geothermal_Vienna'

################# Start of user inputs ##########
sampled_params_path = data_root / 'raw' / 'from_Petrel' / '260904_realization_sampled_parameters.csv' # Petrel sampled parameters, with Temp_grad (C/m) per realization
coor_file_path = data_root / 'processed' / 'post_coor_fault' / 'JD_geothermal_coor&fault&dip.npy' # cell coordinates, depth Z (m) in column 2
save_folder_path = data_root / 'processed' / 'pre_properties' # TEMP files are written here and replace the existing ones
temp_surface = 11 # TEMP = Temp_grad * Z + temp_surface (C)
decimals = 4 # round TEMP to this many decimals to keep the files small; None keeps full precision
case_list = None # e.g. [125, 126] to process only these realizations; None processes every realization with a PORO file in save_folder_path
################## End of user inputs  ##########

temp_grads = load_temp_gradients(sampled_params_path)
depth = load_cell_depths(coor_file_path) # every cell, in the same order as the property files

case_names = sorted([p.name[:-len('_PORO.dat')] for p in save_folder_path.glob('*_PORO.dat')], key=lambda n: int(n.split('_')[-1]))
if case_list is not None:
    case_names = [n for n in case_names if int(n.split('_')[-1]) in case_list]

print(f'Generating TEMP from the sampled temperature gradient for {len(case_names)} realizations')
missing = []
for case_name in case_names:
    if case_name not in temp_grads:
        missing.append(case_name)
        continue
    temp = temperature_from_gradient(depth, temp_grads[case_name], temp_surface)
    if decimals is not None:
        temp = np.round(temp, decimals)
    CMG_format_compress(
        array = temp,
        keyword = 'TEMP',
        max_line_length = 80,
        show_summary = False,
        save_dir = save_folder_path,
        save_name = case_name
    )
    print(f'  {case_name}: Temp_grad = {temp_grads[case_name]:.5f} C/m, TEMP = {temp.min():.2f} - {temp.max():.2f} C')
if missing:
    print(f'Warning: no Temp_grad found for {missing}')

Generating TEMP from the sampled temperature gradient for 96 realizations
Saved compressed TEMP data to: z:\jding\GitHub_Oak\CMG_simulation_processing\01_data\geothermal_Vienna\processed\pre_properties\JD_GEOTHERMAL_112_TEMP.dat
  JD_GEOTHERMAL_112: Temp_grad = 0.02612 C/m, TEMP = 58.87 - 126.44 C
Saved compressed TEMP data to: z:\jding\GitHub_Oak\CMG_simulation_processing\01_data\geothermal_Vienna\processed\pre_properties\JD_GEOTHERMAL_113_TEMP.dat
  JD_GEOTHERMAL_113: Temp_grad = 0.02776 C/m, TEMP = 61.88 - 133.71 C
Saved compressed TEMP data to: z:\jding\GitHub_Oak\CMG_simulation_processing\01_data\geothermal_Vienna\processed\pre_properties\JD_GEOTHERMAL_114_TEMP.dat
  JD_GEOTHERMAL_114: Temp_grad = 0.03208 C/m, TEMP = 69.80 - 152.79 C
Saved compressed TEMP data to: z:\jding\GitHub_Oak\CMG_simulation_processing\01_data\geothermal_Vienna\processed\pre_properties\JD_GEOTHERMAL_115_TEMP.dat
  JD_GEOTHERMAL_115: Temp_grad = 0.03483 C/m, TEMP = 74.84 - 164.94 C
Saved compressed TEMP data

## Step 3: Collect property file names to be used for LSH sampling

In [2]:
import sys
from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root / '02_code' / 'src'))

from pre_extract_filenames_files import extract_sorted_filenames

data_root = project_root / '01_data' / 'geothermal_Vienna'

################# Start of user inputs ##########
property_filenames = extract_sorted_filenames(
        folder_dir = data_root / 'processed' / 'pre_properties', # path to the folder containing the extracted properties
        is_save = True, # whether to save the filenames to a CSV file
        save_dir = data_root / 'processed' / 'pre_property_file_names', # path to the folder where the filenames will be saved
        save_name = '260904_realization_property_filenames.csv', # name of the CSV file where the filenames will be saved
        show_filenames = False # whether to print the filenames to the console
)
print(f'{len(property_filenames)} property files')
################# End of user inputs ############

288 property files
